# Predictive Anayltics: Support Vector Machines with Regression for Community Areas

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
from run_config import PATHS

In [2]:
#TODO: embedding ansehen -> vielleicht austauschen
#TODO: make more time efficient
#TODO: change to thundersvm -> installation umstädlich

In [3]:
GRID_SAMPLE = 4_000_000 # if validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "COMMUNITY_AREAS" # COMMUNITY_AREAS
SPATIAL_ENCODING = "latlong" # options: latlong, onehot
TIME_UNIT = "4H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [4]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv" # same file in full/sample
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [5]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.svm import LinearSVR
# explicitly require this experimental feature
from sklearn.experimental import enable_halving_search_cv # noqa
# now you can import normally from model_selection
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
from joblib import Memory
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder

import networkx as nx
from libpysal.weights import Queen

c:\Users\angel\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preparations

In [6]:
INPUT = PATHS.train_test_dir

In [7]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [8]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [9]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-05-18 20:00:00,5,7,20,8.660254e-01,-5.000000e-01,-0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-01-16 08:00:00,1,4,8,0.000000e+00,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-16 20:00:00,8,6,20,-5.000000e-01,-8.660254e-01,-0.974928,-0.222521,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2026-03-20 16:00:00,3,5,16,8.660254e-01,5.000000e-01,-0.433884,-0.900969,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-10-10 04:00:00,10,5,4,-1.000000e+00,-1.836970e-16,-0.433884,-0.900969,0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156613,2025-12-27 00:00:00,12,6,0,-5.000000e-01,8.660254e-01,-0.974928,-0.222521,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
156614,2025-06-30 16:00:00,6,1,16,5.000000e-01,-8.660254e-01,0.000000,1.000000,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
156615,2025-07-29 20:00:00,7,2,20,1.224647e-16,-1.000000e+00,0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
156616,2025-07-29 20:00:00,7,2,20,1.224647e-16,-1.000000e+00,0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [10]:
#print("Currently working on a sample from all the data due to runtime issues")
#train_df = train_df.sample(n=10_000, random_state=42)

In [11]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-05-18 20:00:00,5,7,20,0.866025,-5.000000e-01,-0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-01-16 08:00:00,1,4,8,0.000000,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-16 20:00:00,8,6,20,-0.500000,-8.660254e-01,-0.974928,-0.222521,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2026-03-20 16:00:00,3,5,16,0.866025,5.000000e-01,-0.433884,-0.900969,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-10-10 04:00:00,10,5,4,-1.000000,-1.836970e-16,-0.433884,-0.900969,0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [12]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [13]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [14]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-05-18 20:00:00,5,7,20,8.660254e-01,-5.000000e-01,-0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-01-16 08:00:00,1,4,8,0.000000e+00,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-16 20:00:00,8,6,20,-5.000000e-01,-8.660254e-01,-0.974928,-0.222521,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2026-03-20 16:00:00,3,5,16,8.660254e-01,5.000000e-01,-0.433884,-0.900969,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-10-10 04:00:00,10,5,4,-1.000000e+00,-1.836970e-16,-0.433884,-0.900969,0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156613,2025-12-27 00:00:00,12,6,0,-5.000000e-01,8.660254e-01,-0.974928,-0.222521,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
156614,2025-06-30 16:00:00,6,1,16,5.000000e-01,-8.660254e-01,0.000000,1.000000,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
156615,2025-07-29 20:00:00,7,2,20,1.224647e-16,-1.000000e+00,0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
156616,2025-07-29 20:00:00,7,2,20,1.224647e-16,-1.000000e+00,0.781831,0.623490,-0.866025,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [16]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: community_area")

    census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
    census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

    census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  


    gdf_proj = gdf.to_crs(epsg=3435)
    gdf["lon"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).x
    gdf["lat"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).y

    tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

    for df in (train_df, val_df, test_df):
        df["community_area"] = df["community_area"].astype(str).str.zfill(2)
        df["lat"] = df["community_area"].map(tract_centroids["lat"])
        df["lon"] = df["community_area"].map(tract_centroids["lon"])

        # sanity check, catch silent join failures early
        n_missing_lat = df["lat"].isna().sum()
        n_missing_lon = df["lon"].isna().sum()
        if n_missing_lat or n_missing_lon:
            print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")


Encoding: latlong and Unit: community_area


In [17]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else:
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Onehot

In [18]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)

Create y

In [19]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [20]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,8.660254e-01,-5.000000e-01,-0.781831,0.623490,-0.866025,0.5,0,14.819037,19.0,4.0,...,0,0,0,0,0,0,1,0.031981,-0.746411,0.664716
1,0.000000e+00,1.000000e+00,0.433884,-0.900969,0.866025,-0.5,0,17.777551,22.0,2.0,...,0,0,1,0,0,0,1,0.031660,-0.746075,0.665108
2,-5.000000e-01,-8.660254e-01,-0.974928,-0.222521,-0.866025,0.5,0,19.240071,26.0,3.0,...,0,1,0,0,0,0,1,0.032110,-0.745862,0.665326
3,8.660254e-01,5.000000e-01,-0.433884,-0.900969,-0.866025,-0.5,0,19.240071,26.0,3.0,...,0,0,0,0,0,0,1,0.032110,-0.745862,0.665326
4,-1.000000e+00,-1.836970e-16,-0.433884,-0.900969,0.866025,0.5,0,15.827760,0.0,2.0,...,0,0,0,0,0,0,1,0.031248,-0.746447,0.664711
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156613,-5.000000e-01,8.660254e-01,-0.974928,-0.222521,0.000000,1.0,0,16.006650,302.0,24.0,...,0,0,1,0,1,0,0,0.029871,-0.743446,0.668128
156614,5.000000e-01,-8.660254e-01,0.000000,1.000000,-0.866025,-0.5,0,16.006650,302.0,24.0,...,1,0,0,0,1,0,0,0.029871,-0.743446,0.668128
156615,1.224647e-16,-1.000000e+00,0.781831,0.623490,-0.866025,0.5,0,12.999588,35.0,7.0,...,1,0,0,0,1,0,0,0.029599,-0.743722,0.667833
156616,1.224647e-16,-1.000000e+00,0.781831,0.623490,-0.866025,0.5,0,17.401160,96.0,24.0,...,1,0,0,0,1,0,0,0.029707,-0.743267,0.668335


### Grid Search

In [21]:
model = SVR()

In [ ]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 1, 'degree': 3, 'epsilon': 1, 'gamma': 'scale', 'kernel': 'poly'}
Best CV score: 0.3961679353349734


### Train Model

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'poly'
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",1
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([ 6.06623197,  1.34785082,  0.92840365, ..., -3.8927612 ,
       -0.86106653, -1.00588888])

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 16.578284127035385
MSE: 4093.5482175240877
RMSE: 63.98084258216742
R2 Score: 0.790151368685712


In [ ]:
# save model
dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/grid_community_svr.joblib']